In [ ]:
#climate_risk_trend_analysis = pd.read_excel('/content/drive/MyDrive/workspace/skyvidya/00_pocs_ingestao_ciencia/mdr/03_gold/climate_risk_trend_analysis_v2.xlsx')
#print(climate_risk_trend_analysis.shape);climate_risk_trend_analysis.head()

In [ ]:
#climate_risk_trend_analysis = gpd.read_parquet('/content/drive/MyDrive/workspace/skyvidya/00_pocs_ingestao_ciencia/mdr/03_gold/climate_risk_trend_analysis.geoparquet')
#print(climate_risk_trend_analysis.shape);climate_risk_trend_analysis.head()

In [ ]:
#climate_risk_trend_analysis[climate_risk_trend_analysis['CD_MUN']=='4314902']

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:

# -*- coding: utf-8 -*-
"""
trend_analysis.py

Este script realiza a análise de tendência de desastres, calcula um score de risco
agregado (MCDA) e identifica a principal ameaça geohidrológica para cada município.
Versão final, que foca a análise da principal ameaça em uma lista predefinida
de códigos COBRADE e opera exclusivamente com dados GeoParquet.
"""

import pandas as pd
import geopandas as gpd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import os
import logging
import re

# --- Dicionário de Mapeamento COBRADE ---
# Mapeia códigos COBRADE para nomes legíveis.
COBRADE_MAP = {
    '11321': 'Deslizamentos',
    '11331': 'Corridas de Massa (Solo/Lama)',
    '11332': 'Corridas de Massa (Rocha/Detrito)',
    '12100': 'Inundações',
    '12200': 'Enxurradas',
    '12300': 'Alagamentos',
    '13111': 'Estiagem',
    '13112': 'Seca',
    '13213': 'Tempestade de Granizo',
    '13214': 'Chuvas Intensas',
    '13215': 'Vendavais',
    '24200': 'Rompimento de Barragem',
    # Adicionar outros códigos se necessário
}

# --- Lista de Códigos de Ameaça para Análise ---
# Apenas os códigos COBRADE listados aqui serão considerados para
# a identificação da "principal ameaça".
GEOHIDROLOGICAL_COBRADES = [
    '11321', # Deslizamentos
    '11331', # Corridas de Massa - Solo/Lama
    '11332', # Corridas de Massa - Rocha/detrito
    '12100', # Inundações
    '12200', # Enxurradas
    '12300', # Alagamentos
    '13213', # Tempestade Local/Convectiva - Granizo
    '13214', # Tempestade Local/Convectiva - Chuvas Intensas
    '13215', # Tempestade Local/Convectiva - Vendaval
    '24200', # Rompimento/colapso de barragens
]

# --- Configurações (Idealmente viriam de um arquivo de configuração) ---
# INPUT_LISA_GEOPARQUET_PATH = '/content/drive/MyDrive/workspace/skyvidya/00_pocs_ingestao_ciencia/mdr/02_silver/climate_risk_lisa_analysis.geoparquet'
# OUTPUT_FINAL_GEOPARQUET_PATH = '/content/drive/MyDrive/workspace/skyvidya/00_pocs_ingestao_ciencia/mdr/03_gold/climate_risk_final_analysis.geoparquet'
# OUTPUT_FINAL_EXCEL_PATH = '/content/drive/MyDrive/workspace/skyvidya/00_pocs_ingestao_ciencia/mdr/03_gold/climate_risk_final_analysis.xlsx'

COLUMNS_FOR_RISK_SCORE_MCDA = [
    'HISTORIC_COUNT', 'LAST10_YEARS_COUNT', 'LAST05_YEARS_COUNT', 'LAST02_YEARS_COUNT',
    'HISTORIC_COUNT_POR_10K_HAB', 'LAST10_YEARS_COUNT_POR_10K_HAB', 'LAST05_YEARS_COUNT_POR_10K_HAB', 'LAST02_YEARS_COUNT_POR_10K_HAB',
    # Adicionar aqui outros critérios MCDA conforme necessário
]
# --------------------------------------------------------------------------

logging.basicConfig(filename='trend_analysis_log.log', level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s')

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="geopandas")


def identify_main_threat(df, relevant_cobrade_codes, timeframe_prefix='LAST10_YEARS_COUNT', cobrade_map=None):
    """
    Identifica a principal ameaça para cada município DENTRO de uma lista
    predefinida de códigos COBRADE.

    Args:
        df (pd.DataFrame): DataFrame de entrada.
        relevant_cobrade_codes (list): Lista de strings com os códigos COBRADE a serem considerados.
        timeframe_prefix (str): O prefixo da coluna a ser usado para determinar a frequência.
        cobrade_map (dict): Dicionário para mapear códigos COBRADE para nomes legíveis.

    Returns:
        pd.DataFrame: DataFrame com a coluna 'principal_ameaca' adicionada.
    """
    if cobrade_map is None:
        cobrade_map = COBRADE_MAP

    logging.info(f"Identificando a principal ameaça geohidrológica com base no prefixo: {timeframe_prefix}")
    print(f"Identificando a principal ameaça geohidrológica com base no prefixo: {timeframe_prefix}...")

    # Constrói a lista de colunas a serem consideradas a partir da lista de códigos relevantes
    threat_cols_to_check = [f"{timeframe_prefix}_{code}" for code in relevant_cobrade_codes]

    # Filtra para apenas as colunas que realmente existem no DataFrame
    existing_threat_cols = [col for col in threat_cols_to_check if col in df.columns]

    if not existing_threat_cols:
        msg = f"Nenhuma coluna de ameaça geohidrológica encontrada com o prefixo '{timeframe_prefix}' e códigos relevantes. Não foi possível identificar a principal ameaça."
        logging.error(msg)
        print(f"ERRO: {msg}")
        df['principal_ameaca'] = "N/A (Faltam colunas de ameaça)"
        return df

    print(f"Colunas de ameaça geohidrológica consideradas: {existing_threat_cols}")
    threat_data = df[existing_threat_cols].copy()

    threat_data.fillna(0, inplace=True)

    max_threat_col = threat_data.idxmax(axis=1)
    is_threat_present = threat_data.max(axis=1) > 0
    cobrade_codes = max_threat_col.str.rsplit('_', n=1).str[-1]

    threat_names = cobrade_codes.map(cobrade_map).fillna(cobrade_codes.apply(lambda code: f"Ameaça Desconhecida (Cod: {code})"))

    df['principal_ameaca'] = np.where(is_threat_present, threat_names, "Nenhuma Ameaça Dominante")

    logging.info("Identificação da principal ameaça concluída.")
    print("Identificação da principal ameaça concluída.")
    return df


def calculate_mcda_risk_score(df, column_names_config, score_col_name, category_col_name):
    """
    Calcula um score de risco agregado (MCDA) e o categoriza.
    """
    logging.info(f"Calculando score de risco MCDA '{score_col_name}'...")
    print(f"Calculando score de risco MCDA '{score_col_name}'...")

    existing_criteria = [col for col in column_names_config if col in df.columns]

    if not existing_criteria:
        msg = "Nenhuma coluna válida (critério) encontrada no DataFrame para calcular o score de risco MCDA."
        logging.error(msg)
        print(f"ERRO: {msg}")
        df[score_col_name] = np.nan
        df[category_col_name] = "N/A (Erro nos critérios)"
        return df

    missing_criteria = [col for col in column_names_config if col not in df.columns]
    if missing_criteria:
        msg = f"AVISO: Os seguintes critérios MCDA não foram encontrados: {missing_criteria}."
        logging.warning(msg)
        print(msg)

    print(f"Critérios MCDA utilizados: {existing_criteria}")
    score_data = df[existing_criteria].copy()
    score_data.fillna(0, inplace=True)

    scaler = MinMaxScaler()
    scaled_data = scaler.fit_transform(score_data)

    df[score_col_name] = scaled_data.mean(axis=1)

    try:
        df[category_col_name] = pd.qcut(df[score_col_name], 5, labels=['Muito Baixo', 'Baixo', 'Médio', 'Alto', 'Muito Alto'], duplicates='drop')
    except Exception as e:
        msg = f"AVISO: pd.qcut falhou ({e}). Tentando pd.cut."
        logging.warning(msg)
        print(msg)
        try:
            min_val, max_val = df[score_col_name].min(), df[score_col_name].max()
            if pd.isna(min_val) or min_val == max_val:
                df[category_col_name] = "Indeterminado"
            else:
                bins = np.linspace(min_val, max_val, 6)
                if len(np.unique(bins)) < 2:
                     bins = 5
                df[category_col_name] = pd.cut(df[score_col_name], bins=bins, labels=['Muito Baixo', 'Baixo', 'Médio', 'Alto', 'Muito Alto'], include_lowest=True, duplicates='drop')
        except Exception as e_cut:
            msg_cut = f"ERRO: Falha ao usar pd.cut: {e_cut}."
            logging.error(msg_cut)
            print(msg_cut)
            df[category_col_name] = "Erro na Categorização"

    logging.info(f"Score e categoria MCDA calculados.")
    print(f"Score e categoria MCDA calculados.")
    return df

def analyze_disaster_trend(df, col_10yr, col_05yr, col_02yr, trend_col_name):
    """
    Calcula a tendência de desastres comparando o crescimento em diferentes períodos.
    """
    logging.info(f"Analisando tendência de desastres para '{trend_col_name}'.")
    print(f"Analisando tendência de desastres para '{trend_col_name}'...")

    required_cols = [col_10yr, col_05yr, col_02yr]
    if not all(col in df.columns for col in required_cols):
        missing = [col for col in required_cols if col not in df.columns]
        msg = f"ERRO: Colunas para análise de tendência não encontradas: {missing}."
        logging.error(msg)
        print(msg)
        df[trend_col_name] = "N/A (Erro nos dados de entrada)"
        return df

    df_trend = df.copy()
    for col in required_cols:
        df_trend[col] = pd.to_numeric(df_trend[col], errors='coerce').fillna(0)

    df_trend['Crescimento_10a_5a_Atras'] = df_trend[col_10yr] - df_trend[col_05yr]
    df_trend['Crescimento_05a_2a_Atras'] = df_trend[col_05yr] - df_trend[col_02yr]

    def determinar_tendencia(row):
        if row['Crescimento_05a_2a_Atras'] > row['Crescimento_10a_5a_Atras']:
            return 'Crescente'
        elif row['Crescimento_05a_2a_Atras'] < row['Crescimento_10a_5a_Atras']:
            return 'Decrescente'
        else:
            return 'Estável'

    df_trend[trend_col_name] = df_trend.apply(determinar_tendencia, axis=1)
    df_trend = df_trend.drop(columns=['Crescimento_10a_5a_Atras', 'Crescimento_05a_2a_Atras'], errors='ignore')

    logging.info(f"Análise de tendência para '{trend_col_name}' concluída.")
    print(f"Análise de tendência para '{trend_col_name}' concluída.")
    return df_trend


def main_trend_analysis(config):
    """
    Função principal para executar o pipeline de análise de tendência.
    """
    logging.info("--- Iniciando Pipeline de Análise de Tendência (MCDA Aprimorada) ---")
    print("--- Iniciando Pipeline de Análise de Tendência (MCDA Aprimorada) ---")

    input_path = config['INPUT_LISA_GEOPARQUET_PATH']
    output_path_gdf = config['OUTPUT_FINAL_GEOPARQUET_PATH']
    output_path_excel = config.get('OUTPUT_FINAL_EXCEL_PATH')

    logging.info(f"Carregando dados de: {input_path}")
    print(f"Carregando dados de: {input_path}...")
    try:
        if not input_path.endswith('.geoparquet'):
            raise ValueError(f"Formato do arquivo de entrada inválido: '{input_path}'. Esperado .geoparquet")
        gdf = gpd.read_parquet(input_path)
    except Exception as e:
        logging.error(f"Erro ao carregar o arquivo de entrada: {e}")
        print(f"ERRO ao carregar o arquivo de entrada: {e}")
        return

    if gdf.empty:
        print(f"ERRO: GeoDataFrame de entrada está vazio. Abortando.")
        return
    logging.info(f"Dados carregados: {gdf.shape[0]} linhas, {gdf.shape[1]} colunas.")
    print(f"Dados carregados: {gdf.shape[0]} linhas, {gdf.shape[1]} colunas.")

    # Passar a lista de códigos relevantes para a função
    gdf = identify_main_threat(gdf,
                               relevant_cobrade_codes=config.get('GEOHIDROLOGICAL_COBRADES', []),
                               timeframe_prefix='LAST10_YEARS_COUNT',
                               cobrade_map=COBRADE_MAP)

    gdf = calculate_mcda_risk_score(gdf,
                               config.get('COLUMNS_FOR_RISK_SCORE_MCDA', []),
                               score_col_name='Risco_Ampliado_MCDA_Score',
                               category_col_name='Risco_Ampliado_MCDA_Cat')

    gdf = analyze_disaster_trend(gdf,
                                 'LAST10_YEARS_COUNT',
                                 'LAST05_YEARS_COUNT',
                                 'LAST02_YEARS_COUNT',
                                 'Tendencia_Eventos_Climaticos_Extremos')

    print(f"\nSalvando GeoDataFrame com análise final em: {output_path_gdf}...")
    try:
        gdf.to_parquet(output_path_gdf)
        print(f"SUCESSO: GeoDataFrame (análise final) salvo em: {output_path_gdf}")
    except Exception as e:
        print(f"ERRO ao salvar o GeoDataFrame (análise final): {e}")

    if output_path_excel:
        print(f"Salvando versão Excel (sem geometria) em: {output_path_excel}...")
        try:
            df_excel = gdf.drop(columns=['geometry'], errors='ignore')
            df_excel.to_excel(output_path_excel, index=False)
            print(f"SUCESSO: Versão Excel salva em: {output_path_excel}")
        except Exception as e:
            print(f"ERRO ao salvar a versão Excel: {e}")

    print("--- Pipeline de Análise de Tendência (MCDA Aprimorada) Concluído ---")


if __name__ == '__main__':
    BASE_PROJECT_PATH = '/content/drive/MyDrive/workspace/skyvidya/00_pocs_ingestao_ciencia/mdr'

    config_trend_analysis = {
        'INPUT_LISA_GEOPARQUET_PATH': os.path.join(BASE_PROJECT_PATH, '02_silver/climate_risk_lisa_analysis.geoparquet'),
        'OUTPUT_FINAL_GEOPARQUET_PATH': os.path.join(BASE_PROJECT_PATH, '03_gold/climate_risk_final_analysis.geoparquet'),
        'OUTPUT_FINAL_EXCEL_PATH': os.path.join(BASE_PROJECT_PATH, '03_gold/climate_risk_final_analysis.xlsx'),
        'COLUMNS_FOR_RISK_SCORE_MCDA': COLUMNS_FOR_RISK_SCORE_MCDA,
        'GEOHIDROLOGICAL_COBRADES': GEOHIDROLOGICAL_COBRADES # Passando a lista de códigos para a configuração
    }

    os.makedirs(os.path.dirname(config_trend_analysis['OUTPUT_FINAL_GEOPARQUET_PATH']), exist_ok=True)
    if config_trend_analysis.get('OUTPUT_FINAL_EXCEL_PATH'):
         os.makedirs(os.path.dirname(config_trend_analysis['OUTPUT_FINAL_EXCEL_PATH']), exist_ok=True)

    main_trend_analysis(config_trend_analysis)
    #print("Exemplo de execução do main_trend_analysis (comentado).")
    #print("Certifique-se que o arquivo de entrada é um GeoParquet.")
    #print("Descomente a linha `main_trend_analysis(config_trend_analysis)` para executar.")

--- Iniciando Pipeline de Análise de Tendência (MCDA Aprimorada) ---
Carregando dados de: /content/drive/MyDrive/workspace/skyvidya/00_pocs_ingestao_ciencia/mdr/02_silver/climate_risk_lisa_analysis.geoparquet...
Dados carregados: 499 linhas, 1057 colunas.
Identificando a principal ameaça geohidrológica com base no prefixo: LAST10_YEARS_COUNT...
Colunas de ameaça geohidrológica consideradas: ['LAST10_YEARS_COUNT_11321', 'LAST10_YEARS_COUNT_11331', 'LAST10_YEARS_COUNT_11332', 'LAST10_YEARS_COUNT_12100', 'LAST10_YEARS_COUNT_12200', 'LAST10_YEARS_COUNT_12300', 'LAST10_YEARS_COUNT_13213', 'LAST10_YEARS_COUNT_13214', 'LAST10_YEARS_COUNT_13215', 'LAST10_YEARS_COUNT_24200']
Identificação da principal ameaça concluída.
Calculando score de risco MCDA 'Risco_Ampliado_MCDA_Score'...
Critérios MCDA utilizados: ['HISTORIC_COUNT', 'LAST10_YEARS_COUNT', 'LAST05_YEARS_COUNT', 'LAST02_YEARS_COUNT', 'HISTORIC_COUNT_POR_10K_HAB', 'LAST10_YEARS_COUNT_POR_10K_HAB', 'LAST05_YEARS_COUNT_POR_10K_HAB', 'LAST02_

In [ ]:
climate_risk_trend_analysis = gpd.read_parquet('/content/drive/MyDrive/workspace/skyvidya/00_pocs_ingestao_ciencia/mdr/03_gold/climate_risk_final_analysis.geoparquet')
print(climate_risk_trend_analysis.shape);climate_risk_trend_analysis.head()

(499, 1061)


,CD_MUN,geometry,CD_RGI,NM_RGI,CD_RGINT,NM_RGINT,CD_UF,NM_UF,SIGLA_UF,CD_REGIA,...,LAST02_YEARS_COUNT_POR_10K_HAB_24200_lisa_I,LAST02_YEARS_COUNT_POR_10K_HAB_24200_lisa_p_sim,LAST02_YEARS_COUNT_POR_10K_HAB_24200_lisa_sig,LAST02_YEARS_COUNT_POR_10K_HAB_24200_lisa_q,LAST02_YEARS_COUNT_POR_10K_HAB_24200_lisa_labels,w_LAST02_YEARS_COUNT_POR_10K_HAB_24200,principal_ameaca,Risco_Ampliado_MCDA_Score,Risco_Ampliado_MCDA_Cat,Tendencia_Eventos_Climaticos_Extremos
0,4300001,"POLYGON ((-52.62752 -32.15022, -52.62816 -32.1...",N/A,N/A,N/A,None,43,Rio Grande do Sul,RS,4,...,NaN,NaN,False,N/A,0 ns,NaN,Nenhuma Ameaça Dominante,0.000000,Muito Baixo,Estável
1,4300002,"POLYGON ((-51.31995 -30.14806, -51.31993 -30.1...",N/A,N/A,N/A,None,43,Rio Grande do Sul,RS,4,...,NaN,NaN,False,N/A,0 ns,NaN,Nenhuma Ameaça Dominante,0.000000,Muito Baixo,Estável
2,4300034,"POLYGON ((-54.21884 -31.82901, -54.23601 -31.8...",430010,Bagé,4302,Pelotas,43,Rio Grande do Sul,RS,4,...,NaN,NaN,False,N/A,0 ns,NaN,Chuvas Intensas,0.166939,Baixo,Decrescente
3,4300059,"POLYGON ((-51.98166 -28.23639, -51.98066 -28.2...",430032,Tapejara - Sananduva,4306,Passo Fundo,43,Rio Grande do Sul,RS,4,...,NaN,NaN,False,N/A,0 ns,NaN,Enxurradas,0.180656,Médio,Decrescente
4,4300109,"POLYGON ((-53.26702 -29.78223, -53.26766 -29.7...",430011,Santa Maria,4303,Santa Maria,43,Rio Grande do Sul,RS,4,...,NaN,NaN,False,N/A,0 ns,NaN,Vendavais,0.252149,Alto,Crescente


In [ ]:
climate_risk_trend_analysis.principal_ameaca

,principal_ameaca
0,Nenhuma Ameaça Dominante
1,Nenhuma Ameaça Dominante
2,Chuvas Intensas
3,Enxurradas
4,Vendavais
...,...
494,Chuvas Intensas
495,Chuvas Intensas
496,Chuvas Intensas
497,Chuvas Intensas


In [ ]:
for i in climate_risk_trend_analysis.columns:
  print("'"+i+"'"+",")

'CD_MUN',
'geometry',
'CD_RGI',
'NM_RGI',
'CD_RGINT',
'NM_RGINT',
'CD_UF',
'NM_UF',
'SIGLA_UF',
'CD_REGIA',
'NM_REGIA',
'SIGLA_RG',
'CD_CONCU',
'NM_CONCU',
'AREA_KM2',
'CENSO_2020_POP',
'NM_MUN_SEM_ACENTO',
'HISTORIC_COUNT',
'LAST10_YEARS_COUNT',
'LAST05_YEARS_COUNT',
'LAST02_YEARS_COUNT',
'HISTORIC_COUNT_POR_10K_HAB',
'LAST10_YEARS_COUNT_POR_10K_HAB',
'LAST05_YEARS_COUNT_POR_10K_HAB',
'LAST02_YEARS_COUNT_POR_10K_HAB',
'HISTORIC_COUNT_11110',
'HISTORIC_COUNT_11120',
'HISTORIC_COUNT_11311',
'HISTORIC_COUNT_11312',
'HISTORIC_COUNT_11313',
'HISTORIC_COUNT_11314',
'HISTORIC_COUNT_11321',
'HISTORIC_COUNT_11331',
'HISTORIC_COUNT_11332',
'HISTORIC_COUNT_11340',
'HISTORIC_COUNT_11410',
'HISTORIC_COUNT_11420',
'HISTORIC_COUNT_11431',
'HISTORIC_COUNT_11432',
'HISTORIC_COUNT_11433',
'HISTORIC_COUNT_12100',
'HISTORIC_COUNT_12200',
'HISTORIC_COUNT_12300',
'HISTORIC_COUNT_13111',
'HISTORIC_COUNT_13112',
'HISTORIC_COUNT_13120',
'HISTORIC_COUNT_13211',
'HISTORIC_COUNT_13212',
'HISTORIC_COUNT_13213',
'

In [ ]:
climate_risk_trend_analysis[climate_risk_trend_analysis['NM_MUN_SEM_ACENTO']=='PORTO ALEGRE']

,CD_MUN,geometry,CD_RGI,NM_RGI,CD_RGINT,NM_RGINT,CD_UF,NM_UF,SIGLA_UF,CD_REGIA,...,LAST02_YEARS_COUNT_POR_10K_HAB_24200_lisa_I,LAST02_YEARS_COUNT_POR_10K_HAB_24200_lisa_p_sim,LAST02_YEARS_COUNT_POR_10K_HAB_24200_lisa_sig,LAST02_YEARS_COUNT_POR_10K_HAB_24200_lisa_q,LAST02_YEARS_COUNT_POR_10K_HAB_24200_lisa_labels,w_LAST02_YEARS_COUNT_POR_10K_HAB_24200,principal_ameaca,Risco_Ampliado_MCDA_Score,Risco_Ampliado_MCDA_Cat,Tendencia_Eventos_Climaticos_Extremos
327,4314902,"MULTIPOLYGON (((-51.1645 -30.2661, -51.1646 -3...",430001,Porto Alegre,4301,Porto Alegre,43,Rio Grande do Sul,RS,4,...,NaN,NaN,False,N/A,0 ns,NaN,Chuvas Intensas,0.147714,Baixo,Estável


In [ ]:
climate_risk_trend_analysis.to_file('/content/drive/MyDrive/workspace/skyvidya/00_pocs_ingestao_ciencia/mdr/03_gold/climate_risk_final_analysis.geojson')

In [ ]:
#climate_risk_trend_analysis = pd.read_excel('/content/drive/MyDrive/workspace/skyvidya/00_pocs_ingestao_ciencia/mdr/03_gold/climate_risk_trend_analysis_v2.xlsx')
#print(climate_risk_trend_analysis.shape);climate_risk_trend_analysis.head()

In [ ]:
#print(climate_risk_trend_analysis.shape);climate_risk_trend_analysis.tail()